# 보관용 D-Fire YOLO26n 시험 파이프라인
이 노트북은 이전 시험 이력입니다. 새 본 학습에는 같은 폴더의 dfire_team_training.ipynb를 사용하세요. 새 노트북에 YOLO11n, v2 라벨 정리, 개인별 실행 폴더와 중간 백업이 적용되어 있습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/2026 3학년 2학기 진로탐색')
PROJECT_DIR = DRIVE_ROOT / 'Code' / 'fire-ai'
DFIRE_DRIVE_DIR = DRIVE_ROOT / 'Data' / 'DFire'
DATASET_ZIP = DFIRE_DRIVE_DIR / 'D-Fire.zip'
SPLIT_ZIP = DFIRE_DRIVE_DIR / 'd-fire 텍스트 분할.zip'
LOCAL_ARCHIVES = Path('/content/dfire-archives')
LOCAL_DATASET = Path('/content/fire-ai-data')
LOCAL_RUNS = Path('/content/runs/fire')
DRIVE_RESULTS = DFIRE_DRIVE_DIR / 'training_results'

for path in (PROJECT_DIR, DATASET_ZIP, SPLIT_ZIP):
    assert path.exists(), f'경로를 찾을 수 없습니다: {path}'
print('프로젝트:', PROJECT_DIR)
print('데이터:', DATASET_ZIP)

In [ ]:
%pip install -q -U ultralytics

In [ ]:
import shutil
import subprocess
import sys

LOCAL_ARCHIVES.mkdir(parents=True, exist_ok=True)
local_dataset_zip = LOCAL_ARCHIVES / DATASET_ZIP.name
local_split_zip = LOCAL_ARCHIVES / SPLIT_ZIP.name
if not local_dataset_zip.exists():
    shutil.copy2(DATASET_ZIP, local_dataset_zip)
if not local_split_zip.exists():
    shutil.copy2(SPLIT_ZIP, local_split_zip)

command = [
    sys.executable,
    str(PROJECT_DIR / 'scripts' / 'prepare_dfire.py'),
    '--dataset-zip', str(local_dataset_zip),
    '--split-zip', str(local_split_zip),
    '--output', str(LOCAL_DATASET),
    '--fold', '1',
]
subprocess.run(command, check=True)

In [ ]:
import json

summary = json.loads((LOCAL_DATASET / 'dataset_summary.json').read_text(encoding='utf-8'))
summary

In [ ]:
import random
import cv2
import matplotlib.pyplot as plt

random.seed(42)
samples = random.sample(list((LOCAL_DATASET / 'val' / 'images').glob('*.jpg')), 6)
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, image_path in zip(axes.ravel(), samples):
    image = cv2.imread(str(image_path))
    height, width = image.shape[:2]
    label_path = LOCAL_DATASET / 'val' / 'labels' / f'{image_path.stem}.txt'
    for line in label_path.read_text(encoding='utf-8-sig').splitlines():
        class_id, xc, yc, bw, bh = map(float, line.split())
        x1, y1 = int((xc - bw / 2) * width), int((yc - bh / 2) * height)
        x2, y2 = int((xc + bw / 2) * width), int((yc + bh / 2) * height)
        color = (0, 200, 255) if int(class_id) == 0 else (0, 0, 255)
        cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
        cv2.putText(image, 'smoke' if int(class_id) == 0 else 'fire', (x1, max(18, y1)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    ax.set_title(image_path.name)
    ax.axis('off')
plt.tight_layout()

In [ ]:
import torch
import ultralytics

print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('Ultralytics:', ultralytics.__version__)
assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'

## 3epoch 재현 시험
이미 완료한 시험을 다시 재현해야 할 때만 `RUN_SMOKE_TEST=True`로 바꿉니다.

In [ ]:
from ultralytics import YOLO

RUN_SMOKE_TEST = False
if RUN_SMOKE_TEST:
    model = YOLO('yolo26n.pt')
    model.train(
        data=str(LOCAL_DATASET / 'fire_v001_colab.yaml'),
        epochs=3, imgsz=640, batch=16, patience=3,
        seed=42, deterministic=True, workers=2, device=0,
        project=str(LOCAL_RUNS), name='yolo26n_smoke_test_3ep',
    )

## 5주차 본 학습
라벨 시각화와 데이터 요약을 확인한 뒤에만 `RUN_FULL_TRAIN=True`로 바꿉니다. test 세트는 모델 선택에 사용하지 않습니다.

In [ ]:
RUN_FULL_TRAIN = False
if RUN_FULL_TRAIN:
    model = YOLO('yolo26n.pt')
    model.train(
        data=str(LOCAL_DATASET / 'fire_v001_colab.yaml'),
        epochs=80, imgsz=640, batch=16, patience=15,
        seed=42, deterministic=True, workers=2, device=0,
        project=str(LOCAL_RUNS), name='yolo26n_fire_v001',
    )
    source = LOCAL_RUNS / 'yolo26n_fire_v001'
    destination = DRIVE_RESULTS / 'yolo26n_fire_v001'
    if destination.exists():
        raise FileExistsError(f'기존 결과를 덮어쓰지 않습니다: {destination}')
    shutil.copytree(source, destination)
    shutil.copy2(LOCAL_DATASET / 'dataset_summary.json', destination / 'dataset_summary.json')
    shutil.copy2(LOCAL_DATASET / 'dataset_manifest_v001.csv', destination / 'dataset_manifest_v001.csv')
    shutil.copy2(LOCAL_DATASET / 'split_v001.csv', destination / 'split_v001.csv')

## 고정 test 평가
validation으로 모델과 임계값을 확정한 뒤에만 실행합니다.

In [ ]:
RUN_TEST_EVAL = False
if RUN_TEST_EVAL:
    best_model = DRIVE_RESULTS / 'yolo26n_fire_v001' / 'weights' / 'best.pt'
    assert best_model.is_file(), best_model
    model = YOLO(str(best_model))
    model.val(
        data=str(LOCAL_DATASET / 'fire_v001_colab.yaml'),
        split='test', imgsz=640, batch=16, device=0, plots=True,
        project=str(LOCAL_RUNS), name='test_v001',
    )